In [1]:
from utils import get_gemini_api_key
GEMINI_API_KEY = get_gemini_api_key()
llm_config = {"model": "llama3.1","api_type":"ollama"}

In [2]:
task = '''
        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       '''


In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [4]:
reply = writer.generate_reply(messages=[{"content": task ,"role": "user"}])

In [5]:
print(reply)

{'content': '**Unlocking Endless Possibilities with Hope AI**\n\nImagine an AI that not only understands human emotions but also shares in our optimism. Welcome to Hope AI, a revolutionary technology designed to amplify positivity and inspire hope. This innovative tool uses advanced natural language processing (NLP) to detect and respond to emotional cues, providing support and encouragement when needed most. With Hope AI, users can expect personalized guidance, uplifting conversations, and empowering experiences that foster a brighter outlook on life. Say goodbye to negativity and hello to endless possibilities with the transformative power of Hope AI.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


In [6]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

**Unlocking Endless Possibilities with Hope AI**

Imagine an AI that not only understands human emotions but also shares in our optimism. Welcome to Hope AI, a revolutionary technology designed to amplify positivity and inspire hope. This innovative tool uses advanced natural language processing (NLP) to detect and respond to emotional cues, providing support and encouragement when needed most. With Hope AI, users can expect personalized guidance, uplifting conversations, and empowering experiences that foster a brighter outlook on life. Say goodbye to negativity and hello to endless possibilities with the transformative power of Hope AI.

--------------------------------------------------------------------------------
Critic (to Writer):

In [8]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [10]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [11]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

In [12]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [13]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

In [14]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

**Unlocking Endless Possibilities with Hope AI**

Imagine an AI that not only understands human emotions but also shares in our optimism. Welcome to Hope AI, a revolutionary technology designed to amplify positivity and inspire hope. This innovative tool uses advanced natural language processing (NLP) to detect and respond to emotional cues, providing support and encouragement when needed most. With Hope AI, users can expect personalized guidance, uplifting conversations, and empowering experiences that foster a brighter outlook on life. Say goodbye to negativity and hello to endless possibilities with the transformative power of Hope AI.

--------------------------------------------------------------------------------

******************

In [15]:
print(res.summary)

**Unlocking Human Potential: A New Era with Hope AI**

Hope AI is an innovative technology designed to amplify positivity and provide personalized support for those seeking guidance and encouragement. This advanced tool utilizes natural language processing (NLP) to understand emotional cues, offering tailored recommendations and uplifting conversations. By leveraging the potential of AI, individuals can develop resilience, build confidence, and cultivate a more optimistic outlook on life.

**Disclaimer:** Hope AI is a hypothetical technology meant to illustrate the possibilities of AI-driven support systems. Its capabilities and limitations are subject to ongoing development and research.
